# Polaris API → VictoriaLogs log-coverage

**What this answers:** for every Polaris API, *if something went wrong on this endpoint
tomorrow, would there be a record of it?*

It is a **log-generating harness**, not an API client and not a functional test suite. Its
output is a statement about observability, not about Polaris correctness — and the answer is
"no" for ten endpoints before it starts. Demonstrating that concretely is the point.

Built from `log-coverage/PLAN-log-coverage.md`, which amends
`~/hynix/local-k8s/POLARIS-API-LOG-COVERAGE-NOTEBOOK.md`. Read the plan first; this notebook
does not restate it.

## Before you run this

```bash
kubectl -n logging       port-forward svc/vlsingle-victoria-logs-single-server 9428:9428
kubectl -n datahub-hynix port-forward deploy/fb-polaris-shipper 2020:2020   # metrics only
```

`local` env only. It creates and deletes a catalog, principals, roles, namespaces, tables and
views — **the cleanup DELETEs are themselves part of the test**, so cell 9 is not optional
tidying, it is a measurement.

## Guards this run needs, and why

| guard | why |
|---|---|
| **do not trip the HPA** | Polaris scales to 3 replicas at 80% CPU, all appending to one `polaris.log` (`local-k8s` #8). The ×20 loops below are two orders of magnitude below that. Replica count is recorded before and after; a change **invalidates the run**. |
| **watch the shipper pod** | rule 6's dedup state is per-pod and in memory. A restart mid-run re-logs the first hit per table, so "exactly one" stops being an invariant. |
| **never print a token or a secret** | notebooks get committed. |
| **do not change Polaris configuration** | `local-k8s` #11/#12: it works and ships continuously, and the `quarkus.log.file.enabled=false` discrepancy is recorded as *unexplained, not pursued*. |

## The expected column is not a table anyone typed

`log_coverage.Policy` extracts `polaris_noise_filter` out of the **deployed**
`logging/fb-values.yaml` and runs it under a real Lua interpreter. If that file or a Lua
binary is missing, cell 0 aborts and says the expected column is unavailable — there is
deliberately no Python re-implementation to fall back on.

## 0 — Preflight

Aborts loudly. A coverage matrix taken from a half-working pipeline is worse than none.

In [ ]:
import pathlib, shutil, subprocess, sys, time
from datetime import datetime, timezone

_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from polaris_test_utils import *          # noqa: F401,F403
import polaris_test_utils as ptu
import vlogs
import log_coverage as lc

init_env("local")
require_not_prod("the log-coverage drive creates and deletes catalogs, principals and roles")

RUN = str(int(time.time()))
NB_PRINCIPAL, NB_PROLE = f"nb_{RUN}_principal", f"nb_{RUN}_prole"
STARTED = datetime.now(timezone.utc)
V = vlogs.VLogs(ptu.VLOGS_URL)
PRE, FAIL = {}, []


def sh(*args):
    """kubectl, when it is there. Returns stdout or None -- never raises."""
    if not shutil.which(args[0]):
        return None
    try:
        r = subprocess.run(args, capture_output=True, text=True, timeout=20)
        return r.stdout.strip() if r.returncode == 0 else None
    except Exception:
        return None


try:
    ROOT_TOK = root_token()
    PRE["polaris"] = f"OK   {ptu.POLARIS_URL}"
except Exception as exc:
    ROOT_TOK = None
    FAIL.append(f"Polaris: {type(exc).__name__}: {exc}")

ok, detail = V.ping()
PRE["victorialogs"] = f"{'OK  ' if ok else 'DOWN'} {ptu.VLOGS_URL}  ({detail})"
if not ok:
    FAIL.append(
        f"VictoriaLogs unreachable at {ptu.VLOGS_URL} -- "
        "kubectl -n logging port-forward svc/vlsingle-victoria-logs-single-server 9428:9428"
    )

try:
    POLICY = lc.load_policy(ptu.FB_VALUES_PATH)
    PRE["retention policy"] = f"OK   {POLICY.source}  sha256={POLICY.sha256[:16]}"
except lc.PolicyUnavailable as exc:
    POLICY = None
    FAIL.append(f"expected column UNAVAILABLE: {exc}")

# IS THE POLICY IN THE FILE THE POLICY THAT IS RUNNING? This repo's most expensive
# failure is configuration written and never applied, and this notebook's first run
# (2026-09-04) found exactly that: fb-values.yaml gained polaris_noise_filter at
# 08:26:18Z on 2026-09-03 and the shipper pod had been up since 08:04:06Z, so the
# parser was live, the policy was not, and NOTHING was dropped. Check before driving,
# rather than inferring it from the shape of the results afterwards.
ALLOW_UNDEPLOYED_POLICY = False      # set True to drive anyway, deliberately
POLICY_LIVE, POLICY_LIVE_DETAIL = None, "policy not loaded"
if POLICY is not None:
    _cm = sh("kubectl", "-n", "datahub-hynix", "get", "cm",
             "-l", "app.kubernetes.io/instance=fb-polaris-shipper", "-o", "yaml")
    POLICY_LIVE, POLICY_LIVE_DETAIL = lc.deployed_policy_status(_cm, POLICY)
PRE["policy DEPLOYED?"] = {True: "YES ", False: "NO  ", None: "??  "}[POLICY_LIVE] + POLICY_LIVE_DETAIL.splitlines()[0]
if POLICY_LIVE is False and not ALLOW_UNDEPLOYED_POLICY:
    FAIL.append(
        "THE RETENTION POLICY IS NOT DEPLOYED. A coverage matrix taken now is a "
        "statement about a pipeline nobody is running.\n    " + POLICY_LIVE_DETAIL
    )

# Do numeric LogsQL filters actually match? `fb-values.yaml` sets `type_int_key
# http_status response_size` so they should. This cannot be answered by looking at a
# returned record's JSON type -- VictoriaLogs hands every field back as a string
# whatever it indexed -- so ask the operational question instead.
NUMERIC_STATUS, NUMERIC_DETAIL = (None, "VictoriaLogs unreachable")
if ok:
    NUMERIC_STATUS, NUMERIC_DETAIL = vlogs.numeric_status_filters_work(V)
PRE["numeric status filters"] = (
    {True: "WORK  ", False: "BROKEN", None: "?     "}[NUMERIC_STATUS] + " " + NUMERIC_DETAIL
)

FB_BEFORE, FB_ERR = vlogs.fluentbit_metrics(ptu.FB_METRICS_URL)
PRE["fluent-bit drop_records"] = FB_ERR or f"{vlogs.drop_records_total(FB_BEFORE):,.0f} (before)"

SHIPPER_BEFORE = sh(
    "kubectl", "-n", "datahub-hynix", "get", "pods",
    "-l", "app.kubernetes.io/instance=fb-polaris-shipper",
    "-o", "jsonpath={range .items[*]}{.metadata.name}@{.status.startTime}{end}",
) or "kubectl unavailable -- a shipper restart mid-run will NOT be detectable"
HPA_BEFORE = sh(
    "kubectl", "-n", "datahub-hynix", "get", "hpa",
    "-o", "jsonpath={range .items[*]}{.metadata.name}={.status.currentReplicas}{end}",
) or "kubectl unavailable -- HPA scaling will NOT be detectable"
PRE["shipper pod (before)"] = SHIPPER_BEFORE
PRE["polaris replicas (before)"] = HPA_BEFORE

for k, v in PRE.items():
    print(f"  {k:26} {v}")
if FAIL:
    raise RuntimeError(
        "PREFLIGHT FAILED:\n  - " + "\n  - ".join(FAIL)
    )
print(f"\n  run={RUN}   principal={NB_PRINCIPAL}   started={STARTED.isoformat()}")

## 1 — Which correlation key does this run get?

`logging.requestIdHeaderName: Polaris-Request-Id` **[from values]** says the client can supply
the request id, which would give exact per-call correlation across the access log *and* the
application log lines the request produced. Whether Polaris honours it is **[assumed]** and is
the one thing the plan could not settle from the repo.

Settle it in minute one, not hour three. If it fails, everything below still runs — the
fallback is `user_principal_name`, `%u` from the access log — but the matrix becomes per-run
rather than per-call, and the report has to say so.

In [ ]:
from polaris_rest import PolarisREST

probe_rid = lc.request_id(RUN, 0, "probe")
adm_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK,
                     extra_headers={"Polaris-Request-Id": probe_rid})

# GET /v1/catalogs is rule 7 ("everything else -> keep"), so it is stored on every call --
# the cheapest possible probe that is guaranteed not to be deduplicated or dropped.
r = adm_pc.list_catalogs()
echoed = r.headers.get("Polaris-Request-Id")
adm_pc.extra_headers = {}

poll = V.poll_until(
    vlogs.and_(vlogs.app_polaris(), vlogs.field_eq("mdc.requestId", probe_rid)),
    expect_at_least=1, timeout=45, interval=1, start="15m",
)
CORRELATION = "exact" if poll.satisfied else "fallback"

print(f"  sent     Polaris-Request-Id: {probe_rid}")
print(f"  echoed   Polaris-Request-Id: {echoed}   ({'honoured' if echoed == probe_rid else 'REWRITTEN by the server'})")
print(f"  found in VictoriaLogs: {len(poll)} record(s) after {poll.waited_s:.1f}s")
print(f"\n  CORRELATION MODE = {CORRELATION.upper()}")
if CORRELATION == "fallback":
    print("  -> mdc.requestId does not round-trip. Per-call rows become per-endpoint rows;")
    print("     the run is selected by user_principal_name and the time window instead.")
    print("     This is a finding in its own right -- record it in the report.")
INGEST_LAG_S = poll.waited_s

## 2 — The denominator: three sources, kept apart

`PLAN` §2.2. Polaris 8181/8182 serves **no** OpenAPI document (measured 2026-08-31), so the
spec has to be vendored — run `./fetch_specs.sh` once. The three sources are unequal and are
never collapsed into a percentage:

| source | what it proves | what it cannot |
|---|---|---|
| **spec** | this VERSION names the endpoint | that this BUILD serves it — generic tables and policies are feature-flagged |
| **captured** | this DEPLOYMENT served it | anything about what was never called: a call log is a lower bound |
| **driven** | this notebook calls it | — |

In [ ]:
import api_surface as surf
from iceberg_rest import build_create_table_payload, build_scan_report, build_schema

HERE = pathlib.Path.cwd()
SPEC, SPEC_FILES = lc.spec_inventory(HERE / "spec")
CAPTURED, CAPTURED_FROM = lc.captured_inventory(
    HERE.parent / "diagnostics" / "api-sql-profile" / "reports"
)

if not SPEC:
    print("  spec      NOT VENDORED -- run ./fetch_specs.sh. The 'candidate' column will be")
    print("            absent, and an endpoint this build does not serve cannot be told")
    print("            apart from one nobody has called.")
else:
    for f in SPEC_FILES:
        print(f"  spec      {f['file']:38} {f['paths']:3} paths  sha256={f['sha256'][:12]}")
    print(f"  spec      {len(SPEC)} operations")
print(f"  captured  {len(CAPTURED)} operations from {CAPTURED_FROM.name if CAPTURED_FROM else 'NOTHING -- no parsable matrix report'}")
print(f"  driven    43 operations (api_surface.operations)")

# The verdict is printed here, before anything is driven, because "confirmed gap" is a
# fact about the harness and does not need this run to establish it.
import collections as _c

DENOM = lc.coverage_rows(SPEC, CAPTURED, lc.driven_inventory(surf.operations(
    surf.SurfaceContext(), build_create_table_payload, build_scan_report)))
print("\n  " + "  ".join(f"{k}={v}" for k, v in
                          _c.Counter(r["verdict"] for r in DENOM).most_common()))
_gaps = [r for r in DENOM if r["verdict"] == "confirmed gap"]
if _gaps:
    print("  CONFIRMED GAPS -- this cluster served these and the notebook does not call them:")
    for r in _gaps[:20]:
        print(f"    {r['method']:6} {r['path']}")

## 3 — Happy path, in dependency order

`api_surface.operations()` is the 43-operation surface, in issue order, and the order is
load-bearing: three chains depend on it (namespace create→drop, table create→rename→drop,
view create→rename→drop) and the management block creates a principal, a principal-role and a
catalog-role before granting on them and deleting them.

Driven with `log_coverage.drive_tagged`, **not** `api_surface.drive`: that one is built around
a SQL `Tracer` and aborts when the capture reads nothing, which is correct for the API→SQL
audit and meaningless here.

`elapsed_ms` is measured client-side because there is nowhere else to get it — the access-log
pattern has no `%D`, so **nothing in VictoriaLogs can say how long any request took**. These
numbers are the evidence for adding it.

In [ ]:
from iceberg_rest import IcebergREST

adm_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)
SCHEMA = build_schema([(1, "id", "long", True), (2, "val", "string", False)])

# `apiprofile*` is the stem teardown_fixture sweeps, so keep it -- and build the
# fixture directly rather than via .stamped(), whose timestamp would not be this
# run's and would make a stranded catalog un-attributable.
FX = surf.ProbeFixture(prefix=f"apiprofile{RUN}")
surf.setup_fixture(FX, adm_pc, adm_ic, ptu.BUCKET, ptu.MINIO_ENDPOINT, SCHEMA,
                   build_create_table_payload)
print(f"  fixture ready: {FX.cat}")

IDENTITY, pc, ic, DRIVE_PRINCIPAL, IDENTITY_NOTES = lc.elect_drive_identity(
    adm_pc, adm_ic, ptu.POLARIS_URL, ptu.REALM, NB_PRINCIPAL, NB_PROLE, fixture=FX
)
for n in IDENTITY_NOTES:
    print(f"  identity  {n}")

CTX = FX.context(ic, pc, adm_ic, SCHEMA)
CTX.adm = adm_pc          # so the fixture's own calls are tagged too
OPS = surf.operations(CTX, build_create_table_payload, build_scan_report)

CALLS = lc.drive_tagged(OPS, CTX, RUN)
print(f"\n  drove {len(CALLS)} operations as {DRIVE_PRINCIPAL}")

import collections
print("  status distribution:", dict(collections.Counter(c["status"] for c in CALLS)))
_err = [c for c in CALLS if c["error"]]
if _err:
    print(f"  {len(_err)} HARNESS ERRORS (not refusals -- these are the harness failing):")
    for c in _err[:10]:
        print(f"    {c['label']}: {c['error']}")

## 4 — Negative cases

Rule 3 (`http_status >= 400` → keep) is the pipeline's real safety net — **not** rule 1, since
a failed request is usually not an ERROR log line at all, just an access-log line with a 4xx
and nothing else. These drive it.

The 500 is `create_catalog_no_endpoint` (`error-cases/09_500_null_pointer`): a catalog missing
`endpointInternal` throws a NullPointerException on table create. Cheapest reliable 500 in
this repo, and reversible — the catalog is deleted immediately after. It is here to answer the
plan's highest-value unknown: **do exception stack traces survive the pipeline?**

The timeout case is honest about its own limit. A client-side timeout does not stop Quarkus
completing the response and writing the access-log line, so it cannot produce the blind spot
being described. The real blind spot is a **server** hang, which cannot be provoked from here.

In [ ]:
import requests

NEG = []
seq = 900


def neg(label, fn, method="", path=""):
    global seq
    seq += 1
    row = lc.call_once([pc, ic, adm_pc, adm_ic], RUN, seq, label, fn, method, path)
    NEG.append(row)
    print(f"  {row['status'] if row['status'] is not None else row['error'][:40]:>6}  {label}")
    return row


# 404 -- a table that is not there. Rule 3 keeps EVERY one of these; rule 6 never sees it.
neg("neg.404_missing_table", lambda: ic.load_table(FX.cat, FX.ns, "definitely_not_here"),
    "GET", "")

# 401 -- a real client id with the wrong secret. The token endpoint's SUCCESS is dropped by
# rule 5; its FAILURE is kept by rule 3. So "who failed to authenticate" is answerable and
# "who authenticated" is not.
neg("neg.401_bad_secret", lambda: adm_pc.get_token(ptu.ROOT_CLIENT, "not-the-secret"),
    "POST", "")

# 403 -- a principal with a role and no grants at all.
DENIED = f"nb_{RUN}_denied"
DENIED_ROLE = f"nb_{RUN}_denied_role"
denied_ic = None
try:
    _cid, _sec = lc.provision_run_principal(adm_pc, DENIED, DENIED_ROLE,
                                            fixture=None, service_admin=False)
    _r = adm_pc.get_token(_cid, _sec, scope=f"PRINCIPAL_ROLE:{DENIED_ROLE}")
    if _r.status_code == 200:
        denied_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=_r.json()["access_token"])
        # denied_ic must be tagged too. On 2026-09-04 it was not in the client list, so
        # the 403 went out with no Polaris-Request-Id, could not be found afterwards,
        # and the matrix reported "EXPECTED STORED, ABSENT" for a record that was
        # almost certainly there -- a harness gap wearing a finding's clothes.
        seq += 1
        row = lc.call_once([denied_ic], RUN, seq, "neg.403_no_privilege",
                           lambda: denied_ic.load_table(FX.cat, FX.ns, FX.tbl), "GET", "")
        NEG.append(row)
        print(f"  {row['status']:>6}  neg.403_no_privilege")
    else:
        print(f"  ---     403 case skipped: token for {DENIED} -> {_r.status_code}")
except Exception as exc:
    print(f"  ---     403 case skipped: {type(exc).__name__}: {exc}")

# 409 -- create a namespace that already exists.
neg("neg.409_already_exists", lambda: ic.create_namespace(FX.cat, FX.ns), "POST", "")

# 400 -- a create-table body with no schema.
neg("neg.400_malformed_body", lambda: ic.create_table(FX.cat, FX.ns, {"name": "no_schema"}),
    "POST", "")

# 405 -- a method the endpoint does not implement.
neg("neg.405_method_not_allowed",
    lambda: requests.request("DELETE", f"{ptu.POLARIS_URL}/api/catalog/v1/config",
                             headers=ic._h(), timeout=15),
    "DELETE", "")

# 500 -- NullPointerException on table create against a catalog with no endpointInternal.
NPE_CAT = f"nb{RUN}npe"
try:
    _r = create_catalog_no_endpoint(name=NPE_CAT, token=ROOT_TOK)
    print(f"  {_r.status_code:>6}  neg.500_setup_catalog_without_endpointInternal")
    ic.create_namespace(NPE_CAT, "npe_ns")
    neg("neg.500_null_pointer",
        lambda: ic.create_table(NPE_CAT, "npe_ns",
                                build_create_table_payload("npe_tbl", SCHEMA)),
        "POST", "")
finally:
    try:
        adm_pc.delete_catalog(NPE_CAT, purge=True)
    except Exception as exc:
        print(f"  ---     could not remove {NPE_CAT}: {exc}")

# timeout -- attempted, and expected NOT to reproduce the blind spot. Recorded either way.
neg("neg.client_timeout",
    lambda: requests.get(f"{ptu.POLARIS_URL}/api/management/v1/catalogs",
                         headers=adm_pc._h(), timeout=0.001),
    "GET", "/api/management/v1/catalogs")

# 429 is UNREACHABLE: rateLimiter.type is no-op [from values]. Recorded as not callable,
# with the reason. An honest gap beats a fabricated pass.
NOT_CALLABLE = {
    "429 rate limited": "rateLimiter.type: no-op -- the limiter cannot produce a 429",
    "OPA authorization paths": "opa.enabled: false -- those code paths are dead here",
    "federated catalog endpoints": "no federated catalog is configured on this cluster",
    "a server-side hang": "cannot be provoked from a client; the access log is written when "
                          "the response is, so a hang leaves NO line at all and this pipeline "
                          "is blind to it",
}

## 5 — Policy probes

Deliberate tests of the rules themselves. Every expected count here comes from
`Policy.predict()` — the **deployed** Lua, run in issue order so rule 6's day-buckets build up
exactly as the shipper's do — so the "expected" column cannot drift from what ships.

Two probes the source plan does not have, both from `PLAN` §2.3: a table **rename** and a
credential **reset** are dropped, which the plan's one named asymmetry (principals created
invisibly, deleted visibly) does not cover.

In [ ]:
PROBE_CAT, PROBE_NS, PROBE_TBL, PROBE_VIEW = FX.cat, FX.ns, FX.tbl, FX.view
PROBES, seq = [], 1000


def probe(name, n, fn, method, path_for_prediction):
    """Run `fn` n times, tagged. The expectation is computed in cell 8, not here.

    Deliberately NOT predicted per probe. Rule 6's dedup state is per shipper
    pod and spans the WHOLE run: cell 3 already read `probe_tbl` once, so the
    twenty reads below are the second through twenty-first of the day and the
    honest expectation is ZERO, not one. A per-probe prediction starts from a
    fresh interpreter, would say one, and the mismatch would be reported as a
    pipeline finding when it was an artifact of predicting in the wrong scope.
    """
    global seq
    rows = []
    for i in range(n):
        seq += 1
        rows.append(lc.call_once([pc, ic, adm_pc, adm_ic], RUN, seq, f"probe.{name}",
                                 fn, method, path_for_prediction))
    PROBES.append({"probe": name, "calls": n, "rows": rows})
    codes = sorted({r["status"] for r in rows})
    print(f"  {name:44} calls={n:<3} status={codes}")
    return rows


CATP = f"/api/catalog/v1/{PROBE_CAT}"
MGMTP = "/api/management/v1"

probe("same table GET x20", 20, lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/tables/{PROBE_TBL}")
probe("same table GET ?snapshots=all", 1,
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL, snapshots="all"),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/tables/{PROBE_TBL}?snapshots=all")
probe("missing table GET x20", 20, lambda: ic.load_table(PROBE_CAT, PROBE_NS, "nope"),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/tables/nope")
probe("HEAD the same table", 1, lambda: ic.head_table(PROBE_CAT, PROBE_NS, PROBE_TBL),
      "HEAD", f"{CATP}/namespaces/{PROBE_NS}/tables/{PROBE_TBL}")
probe("table LIST x20", 20, lambda: ic.list_tables(PROBE_CAT, PROBE_NS),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/tables")
probe("successful token x3", 3,
      lambda: adm_pc.get_token(ptu.ROOT_CLIENT, ptu.ROOT_SECRET), "POST",
      "/api/catalog/v1/oauth/tokens")

# The create/delete audit asymmetry, demonstrated end to end.
ASYM = f"nb_{RUN}_asym"
probe("create a principal", 1, lambda: adm_pc.create_principal(ASYM),
      "POST", f"{MGMTP}/principals")
probe("reset that principal's credentials", 1,
      lambda: adm_pc.reset_principal_credentials(ASYM),
      "POST", f"{MGMTP}/principals/{ASYM}/reset")
probe("delete that principal", 1, lambda: adm_pc.delete_principal(ASYM),
      "DELETE", f"{MGMTP}/principals/{ASYM}")

# A rename is a mutation whose path carries no /namespaces/{ns}/tables segment.
RENAMED = f"{PROBE_TBL}_renamed"
probe("rename a table", 1,
      lambda: ic.rename_table(PROBE_CAT, PROBE_NS, PROBE_TBL, PROBE_NS, RENAMED),
      "POST", f"{CATP}/tables/rename")
probe("rename it back", 1,
      lambda: ic.rename_table(PROBE_CAT, PROBE_NS, RENAMED, PROBE_NS, PROBE_TBL),
      "POST", f"{CATP}/tables/rename")

## 6 — Where is a PUT's request body logged?

Rule 4 keeps every PUT/DELETE/PATCH, but **the request body is not in the access log**. A PUT
that changes a role's privileges is recorded as *"a PUT happened to this path, and it returned
200"*. For an audit trail of grants that is the whole point, and it is missing.

So: grant a privilege, then search VictoriaLogs for *any* record carrying the privilege name.
Then look at the other channel. The `eventListener` writes audit events to a PostgreSQL table
(`type: persistence-in-memory-buffer`, 5s flush, 1000-event buffer **[from values]**) — a
second audit channel that does not go through VictoriaLogs at all. If that is where "what
changed" lives, **it changes where you go during an incident**, and the report has to say so.

In [ ]:
GRANTED_PRIV = "TABLE_READ_DATA"
seq += 1
grant_row = lc.call_once([pc, ic, adm_pc, adm_ic], RUN, seq, "probe.grant_privilege",
                         lambda: adm_pc.grant_privilege(FX.cat, "catalog_admin", GRANTED_PRIV),
                         "PUT", f"/api/management/v1/catalogs/{FX.cat}/catalog-roles/catalog_admin/grants")
print(f"  grant {GRANTED_PRIV} -> {grant_row['status']}")

time.sleep(max(2.0, INGEST_LAG_S))
priv_hits = V.query(vlogs.and_(vlogs.app_polaris(), f"_msg:{vlogs.quote(GRANTED_PRIV)}"),
                    start="10m", limit=50)
print(f"  records in VictoriaLogs carrying the privilege NAME: {len(priv_hits)}")
if not priv_hits:
    print("  -> the pipeline cannot answer 'who granted what'. The PUT is stored; the body is not.")

# Polaris creates its objects in POLARIS_SCHEMA, not `public` -- looking there reported
# "no events table in this metastore" on 2026-09-04 for a table that may well exist.
EVENTS = {"reachable": False, "rows": None, "sample": None, "note": None, "schema": None}
try:
    import psycopg2
    with psycopg2.connect(**ptu.PG_CONFIG) as conn:
        EVENTS.update(lc.events_table(conn))
        EVENTS["reachable"] = EVENTS.get("rows") is not None
except Exception as exc:
    EVENTS["note"] = f"{type(exc).__name__}: {exc}"

if EVENTS["reachable"]:
    print(f"  found `events` in schema {EVENTS['schema']}")
    print(f"  eventListener `events` table: {EVENTS['rows']} rows -- a SECOND audit channel,")
    print("  and it does not pass through VictoriaLogs at all.")
    for row in EVENTS["sample"] or []:
        print("   ", {k: str(v)[:60] for k, v in row.items()})
else:
    print(f"  events table not read: {EVENTS['note']}")
    print("  (needs a port-forward: kubectl -n datahub-hynix port-forward "
          "pod/benchmarks-postgresql-postgresql-ha-postgresql-0 5433:5432)")

## 7 — Collection

Ingest is asynchronous, so this polls rather than sleeping and hoping — and it deduplicates by
`mdc.requestId` before counting, because the shipper's tail offset DB lives on an `emptyDir`:
a `helm upgrade` replaces the pod, `Read_from_Head true` re-reads the whole file, and
VictoriaLogs does not deduplicate on ingest. **Records can legitimately appear twice.**

In [ ]:
ALL_CALLS = CALLS + NEG + [r for p in PROBES for r in p["rows"]] + [grant_row]
WINDOW_START = STARTED
WINDOW_END = datetime.now(timezone.utc)

if CORRELATION == "exact":
    run_filter = vlogs.and_(vlogs.app_polaris(), f'"mdc.requestId":"nb-{RUN}-"*')
else:
    run_filter = vlogs.and_(vlogs.app_polaris(),
                            vlogs.field_eq("user_principal_name", DRIVE_PRINCIPAL))

pull = V.settle(run_filter, quiet_for=4, timeout=90, interval=2,
                start=WINDOW_START, end=WINDOW_END, limit=10000)
# Deduplicate REPLAYED RECORDS, not request ids: one call produces many records that
# share a request id (its access-log line plus every application line it emitted), and
# keying on the request id collapses them into one. See vlogs.dedup_replayed.
STORED = vlogs.dedup_replayed(pull.records)
DUPES = len(pull.records) - len(STORED)

BY_RID = {}
for rec in STORED:
    BY_RID.setdefault(rec.get("mdc.requestId"), []).append(rec)

print(f"  calls issued          {len(ALL_CALLS)}")
print(f"  records returned      {len(pull.records)}  (settled after {pull.waited_s:.0f}s)")
print(f"  after replay dedup    {len(STORED)}   [{DUPES} duplicate(s) -- a shipper replay if > 0]")
print(f"  distinct request ids  {len([k for k in BY_RID if k])}")
print(f"  records per call      {len(STORED) / max(1, len(ALL_CALLS)):.1f}")

FB_AFTER, FB_AFTER_ERR = vlogs.fluentbit_metrics(ptu.FB_METRICS_URL)
DROP_DELTA = (None if (FB_ERR or FB_AFTER_ERR)
              else vlogs.drop_records_total(FB_AFTER) - vlogs.drop_records_total(FB_BEFORE))
print(f"  fluent-bit drop delta {DROP_DELTA if DROP_DELTA is not None else FB_AFTER_ERR}")

SHIPPER_AFTER = sh("kubectl", "-n", "datahub-hynix", "get", "pods",
                   "-l", "app.kubernetes.io/instance=fb-polaris-shipper",
                   "-o", "jsonpath={range .items[*]}{.metadata.name}@{.status.startTime}{end}")
HPA_AFTER = sh("kubectl", "-n", "datahub-hynix", "get", "hpa",
               "-o", "jsonpath={range .items[*]}{.metadata.name}={.status.currentReplicas}{end}")
SHIPPER_MOVED = bool(SHIPPER_AFTER) and SHIPPER_AFTER != SHIPPER_BEFORE
HPA_MOVED = bool(HPA_AFTER) and HPA_AFTER != HPA_BEFORE
if SHIPPER_MOVED:
    print("  !! THE SHIPPER RESTARTED MID-RUN. Dedup state is per-pod and in memory, so the")
    print("     'exactly one per day' expectations below are VOID for this run, not failed.")
if HPA_MOVED:
    print("  !! POLARIS SCALED MID-RUN. Multiple replicas append to one log file "
          "(local-k8s #8) -- this run is INVALID.")

## 8 — The coverage matrix

One row per endpoint. `expected` comes from the deployed Lua; `stored` from VictoriaLogs; the
`discrepancy` column is the only one worth reading twice.

A row where expected is *drop* and stored is 0 is **not a pass** in the ordinary sense — it is
the pipeline working exactly as configured, and the configuration being wrong. Those rows are
the finding.

In [ ]:
import pandas as pd

EXPECTED = lc.expected_for(ALL_CALLS, POLICY, user=DRIVE_PRINCIPAL) if POLICY else [None] * len(ALL_CALLS)

rows = []
for call, exp in zip(ALL_CALLS, EXPECTED):
    found = BY_RID.get(call["request_id"], []) if CORRELATION == "exact" else None
    access = [r for r in (found or []) if vlogs.is_access_log(r)]
    rows.append({
        "label": call["label"],
        "method": call["method"],
        "path": (call["actual_path"] or call["path"] or "")[-58:],
        "status": call["status"],
        "ms": call["elapsed_ms"],
        "expected": (exp or {}).get("verdict") if exp else None,
        "stored": len(access) if found is not None else None,
        "app_lines": len(found) - len(access) if found is not None else None,
    })
M = pd.DataFrame(rows)
M["discrepancy"] = [
    "" if r.expected is None or r.stored is None
    else "" if (r.expected == "keep") == (r.stored > 0)
    else ("EXPECTED STORED, ABSENT" if r.expected == "keep" else "EXPECTED DROPPED, PRESENT")
    for r in M.itertuples()
]
pd.set_option("display.width", 200, "display.max_rows", 300, "display.max_colwidth", 60)
print(M.to_string(index=False))

print("\n--- calls the deployed policy says should leave NO record ---")
_drop = M[M.expected == "drop"]
print(f"  {len(_drop)} calls; {int((_drop.stored.fillna(0) > 0).sum())} of them are stored anyway")
print(_drop[["label", "method", "status", "stored"]].to_string(index=False))

print("\n--- discrepancies (expected vs actual) ---")
_d = M[M.discrepancy != ""]
print("none" if _d.empty else _d.to_string(index=False))

print("\n--- probe expectations (predicted across the WHOLE run, in issue order) ---")
EXP_BY_RID = {c["request_id"]: (e or {}).get("verdict") for c, e in zip(ALL_CALLS, EXPECTED)}
for p in PROBES:
    want = sum(1 for row in p["rows"] if EXP_BY_RID.get(row["request_id"]) == "keep")
    got = (sum(len([r for r in BY_RID.get(row["request_id"], []) if vlogs.is_access_log(r)])
               for row in p["rows"]) if CORRELATION == "exact" else None)
    verdict = "" if got is None or got == want else "   <-- MISMATCH"
    print(f"  {p['probe']:44} calls {p['calls']:<3} expected {want:<3} got {got}{verdict}")

print("\n--- WARN / ERROR messages seen (source plan section 8.1) ---")
WARNS = V.query(vlogs.and_(vlogs.app_polaris(), 'level:"WARN" OR level:"ERROR"'),
                start=WINDOW_START, end=WINDOW_END, limit=1000)
DISTINCT_WARNS = {}
for w in WARNS:
    key = (w.get("level"), (w.get("_msg") or "")[:120])
    DISTINCT_WARNS[key] = DISTINCT_WARNS.get(key, 0) + 1
for (lvl, msg), n in sorted(DISTINCT_WARNS.items(), key=lambda kv: -kv[1]):
    print(f"  {lvl:5} x{n:<4} {msg}")
if not DISTINCT_WARNS:
    print("  none -- so the 'Deprecated Config' exclusion still has no message to match on.")

print("\n--- did any record carry an exception / stack trace? ---")
EXC = [r for r in WARNS if "exception" in r or "stackTrace" in str(r)]
print(f"  {len(EXC)} of {len(WARNS)} WARN/ERROR records carry an exception object")
if EXC:
    print("  ->", str(EXC[0].get("exception") or EXC[0].get("_msg"))[:300])
else:
    print("  -> NOT DEMONSTRATED. Either the 500 produced no throwable in the log, or the")
    print("     pipeline drops them. Check the raw file before concluding:")
    print("     kubectl -n datahub-hynix exec deploy/benchmarks-polaris -- "
          "grep -c stackTrace /deployments/logs/polaris.log")

## 9 — Cleanup

The cleanup DELETEs are part of the test — they are driven inside the tagged window, not after it. A failed run must still leave the cluster usable.

In [ ]:
CLEANUP = {}
CLEANUP["fixture"] = surf.teardown_fixture(FX, adm_pc, adm_ic, sweep_stale=False)
CLEANUP["run_principal"] = lc.deprovision_run_principal(adm_pc, NB_PRINCIPAL, NB_PROLE)
CLEANUP["denied_principal"] = lc.deprovision_run_principal(adm_pc, DENIED, DENIED_ROLE)
try:
    adm_pc.delete_catalog(f"nb{RUN}npe", purge=True)
except Exception:
    pass

for k, v in CLEANUP.items():
    print(f"  {k:20} {v}")

try:
    del pull
except NameError:
    pass
import gc
gc.collect()
print("\n  cleaned. Anything left behind is listed above.")

## 10 — Findings

Written to `doc-log-coverage-results.md`, for someone who was not there.

In [ ]:
def q(x):
    return "unknown" if x is None else x


report = f"""# Log coverage — run {RUN}

Generated by `polaris_log_coverage.ipynb` on {STARTED.date().isoformat()}.
Correlation mode **{CORRELATION}**; drove as **{DRIVE_PRINCIPAL}**.
Expected column computed from `{POLICY.source if POLICY else 'UNAVAILABLE'}`
(sha256 `{POLICY.sha256[:16] if POLICY else '-'}`).

## Validity

| | |
|---|---|
| shipper pod before | `{SHIPPER_BEFORE}` |
| shipper pod after | `{q(SHIPPER_AFTER)}` |
| Polaris replicas before / after | `{HPA_BEFORE}` / `{q(HPA_AFTER)}` |
| duplicate records (shipper replay) | {DUPES} |
| ingest lag, first record | {INGEST_LAG_S:.1f}s |
| numeric `http_status` filters | {q(NUMERIC_DETAIL)} |
| retention policy deployed? | {q(POLICY_LIVE)} — {POLICY_LIVE_DETAIL.splitlines()[0]} |
| records per call | {len(STORED) / max(1, len(ALL_CALLS)):.1f} |

{"**THIS RUN IS INVALID** — Polaris scaled mid-run." if HPA_MOVED else ""}
{"**Dedup expectations are VOID** — the shipper restarted mid-run." if SHIPPER_MOVED else ""}

## The seven questions

1. **Distinct WARN messages** — {len(DISTINCT_WARNS)} seen:
{chr(10).join(f"   - `{lvl}` x{n}: {msg}" for (lvl, msg), n in DISTINCT_WARNS.items()) or "   - none, so the Deprecated Config exclusion still has nothing to match narrowly."}
2. **Where a PUT's body is logged** — {len(priv_hits)} record(s) in VictoriaLogs carry the
   granted privilege name. eventListener `events` table: {"%s rows in schema %s" % (EVENTS["rows"], EVENTS["schema"]) if EVENTS["reachable"] else EVENTS["note"]}.
3. **Do stack traces survive?** — {len(EXC)} of {len(WARNS)} WARN/ERROR records carried an
   exception object.
4. **Does `mdc.requestId` join reliably?** — {CORRELATION}; {len([k for k in BY_RID if k])} distinct
   ids recovered from {len(ALL_CALLS)} calls.
5. **The create/delete audit asymmetry** — {int((M.expected == "drop").sum())} of {len(M)} calls in this
   run produce no record at all.
6. **Volume** — {len(STORED)} records stored for {len(ALL_CALLS)} calls;
   `fluentbit_filter_drop_records_total` delta {q(DROP_DELTA)}.
7. **Client-side latency** (there is no `%D`, so this is the only source):
   median {M.ms.median():.0f} ms, p95 {M.ms.quantile(0.95):.0f} ms, max {M.ms.max():.0f} ms.

## Not callable on this cluster, with the reason

{chr(10).join(f"- **{k}** — {v}" for k, v in NOT_CALLABLE.items())}

## Coverage matrix

```
{M.to_string(index=False)}
```
"""
out = pathlib.Path("doc-log-coverage-results.md")
out.write_text(report, encoding="utf-8")
print(f"  wrote {out} ({len(report):,} bytes)")
print("\n  Next: update MEMORY.md 'Now', put the numbers in .memory/roadmap.md, anything")
print("  not to be trusted in .memory/active-issues.md, and the wrong turns in")
print("  .memory/sessions/<date>-log-coverage.md. Then commit -- one task, one commit.")